# TabICL L4 (Colab) — exp_070 fold0
**런타임 → 런타임 유형 변경 → L4 GPU** 선택. 좌측 🔑 Secrets에 **KAGGLE_USERNAME · KAGGLE_KEY** 등록(노트북 액세스 ON) 후 위에서부터 실행.
OOF는 마지막 셀에서 다운로드 → `/teamspace .../experiments/oof/`에 복사. 실행법 SSOT=docs/wiki/colab_jobs.md

In [ ]:
# 1) 설치 (tabicl + 우리 src 의존)
!pip install -q tabicl hydra-core omegaconf python-dotenv scikit-learn pandas
import torch; print('CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))  # L4 확인

In [ ]:
# 2) Kaggle 인증 — Colab Secrets(KAGGLE_USERNAME/KAGGLE_KEY) 사용 (kaggle.json 업로드 불요)
from google.colab import userdata
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)
json.dump({'username': userdata.get('KAGGLE_USERNAME'), 'key': userdata.get('KAGGLE_KEY')},
          open('/root/.kaggle/kaggle.json','w'))
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle 인증 OK (Colab Secrets)')

In [ ]:
# 3) 데이터(대회) + 코드(f1-pit-src) 다운로드
!kaggle competitions download -c playground-series-s6e5 -p /content/comp -q && cd /content/comp && unzip -oq '*.zip'
!kaggle datasets download -d buzziru/f1-pit-src -p /content/srcd -q && cd /content/srcd && unzip -oq '*.zip'
!ls /content/comp && echo '---' && ls /content/srcd

In [ ]:
# 4) src import + 경로 override (Colab 경로로)
import sys; sys.path.insert(0, '/content/srcd')
from pathlib import Path
from src import config
config.TRAIN_PATH = Path('/content/comp/train.csv')
config.TEST_PATH = Path('/content/comp/test.csv')
config.SAMPLE_SUBMISSION_PATH = Path('/content/comp/sample_submission.csv')
out = Path('/content/out'); config.OOF_DIR=out/'oof'; config.SUBMISSION_DIR=out/'submissions'; config.LOG_DIR=out/'logs'
for d in [config.OOF_DIR, config.SUBMISSION_DIR, config.LOG_DIR]: d.mkdir(parents=True, exist_ok=True)
from src.train_tabicl import run
print('import OK')

In [ ]:
# 4.5) 소규모 fast-fail (10k — L4 24GB 메모리·API 검증)
import pandas as pd, time
from tabicl import TabICLClassifier
_tr=pd.read_csv(config.TRAIN_PATH).sample(10000, random_state=42)
_y=_tr['PitNextLap']; _X=_tr.drop(columns=['id','PitNextLap'])
for c in _X.select_dtypes('object').columns: _X[c]=_X[c].astype('category').cat.codes
t0=time.time(); m=TabICLClassifier(device='cuda', n_estimators=8, batch_size=2, offload_mode='auto', random_state=42)
m.fit(_X,_y); print(f'10k OK {time.time()-t0:.0f}s | L4면 full 진행')

In [ ]:
# 5) full fold0 run (base raw, augment False)
from omegaconf import OmegaConf
import time
CONF = Path('/content/srcd/conf')
cfg = OmegaConf.create({
    'exp_id': 'exp_070_tabicl_l4_fold0',
    'notes': 'TabICL L4 fold0: base(raw), offload auto',
    'use_wandb': False, 'max_folds': 1,
    'model': OmegaConf.load(CONF/'model'/'tabicl.yaml'),
    'features': OmegaConf.load(CONF/'features'/'base.yaml'),
    'augment': {'enabled': False, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))
t0=time.time(); result=run(cfg); print(result, f'{time.time()-t0:.0f}s')
print('fold0 AUC =', result.get('fold_scores',[None])[0])

In [ ]:
# 6) OOF 다운로드 → /teamspace .../experiments/oof/ 에 복사
from google.colab import files
files.download(str(config.OOF_DIR/'exp_070_tabicl_l4_fold0.csv'))
files.download(str(config.SUBMISSION_DIR/'exp_070_tabicl_l4_fold0.csv'))